In [3]:
import argparse
import copy
import os
import time
import json
from collections import defaultdict
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torchvision import transforms, datasets, models
import timm
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from ptflops import get_model_complexity_info
import pandas as pd
from tqdm import tqdm

In [4]:
# --------------------------
# Utilities
# --------------------------
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def count_nonzero_params(model: torch.nn.Module) -> Tuple[int, int]:
    total = 0
    nonzero = 0
    for p in model.parameters():
        num = p.numel()
        total += num
        nonzero += (p.abs() > 1e-8).sum().item()
    return total, nonzero

def measure_inference_time(model, device, input_size=(3,224,224), n_runs=100, batch_size=1):
    model.eval()
    dummy = torch.randn((batch_size,)+input_size).to(device)
    # warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy)
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.time()
            _ = model(dummy)
            t1 = time.time()
            times.append(t1 - t0)
    mean_ms = 1000.0 * np.mean(times)
    std_ms = 1000.0 * np.std(times)
    return mean_ms, std_ms

def compute_flops(model, input_res=(3,224,224)):
    # ptflops wants a function that returns input shape tuple minus batch dim
    try:
        macs, params = get_model_complexity_info(model, input_res[1:], as_strings=False, print_per_layer_stat=False)
        # ptflops returns MACs (multiply-adds). Conventionally GFLOPs = MACs / 1e9
        gflops = macs / 1e9
        return gflops, params
    except Exception as e:
        print("ptflops failed:", e)
        return None, None

In [5]:
# --------------------------
# Dataset loaders
# --------------------------
def make_dataloaders(data_dir, image_size=224, batch_size=32, num_workers=4):
    transform_train = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    transform_eval = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    train_dir = os.path.join(data_dir, "train")
    val_dir   = os.path.join(data_dir, "val")
    test_dir  = os.path.join(data_dir, "test")
    train_ds = datasets.ImageFolder(train_dir, transform=transform_train)
    val_ds = datasets.ImageFolder(val_dir, transform=transform_eval)
    test_ds = datasets.ImageFolder(test_dir, transform=transform_eval)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader, len(train_ds.classes)


In [6]:
# --------------------------
# Training / Evaluation
# --------------------------
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    preds = []
    trues = []
    for x,y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        preds += out.argmax(dim=1).detach().cpu().tolist()
        trues += y.detach().cpu().tolist()
    loss = running_loss / len(loader.dataset)
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average='macro')
    return loss, acc, f1

def eval_model(model, loader, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            preds += out.argmax(dim=1).cpu().tolist()
            trues += y.cpu().tolist()
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average='macro')
    prec = precision_score(trues, preds, average='macro')
    rec = recall_score(trues, preds, average='macro')
    return {'acc': acc, 'f1': f1, 'prec': prec, 'rec': rec}


In [7]:

# --------------------------
# Knowledge Distillation loss
# --------------------------
class KDLoss(nn.Module):
    def __init__(self, temperature=10.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.kl = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, targets):
        T = self.temperature
        p_s = nn.functional.log_softmax(student_logits / T, dim=1)
        p_t = nn.functional.softmax(teacher_logits / T, dim=1)
        loss_soft = self.kl(p_s, p_t) * (T * T)
        loss_hard = self.ce(student_logits, targets)
        return self.alpha * loss_soft + (1.0 - self.alpha) * loss_hard

def train_kd_epoch(student, teacher, loader, kd_loss_fn, optimizer, device):
    student.train()
    teacher.eval()
    running_loss = 0.0
    preds, trues = [], []
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        s_logits = student(x)
        with torch.no_grad():
            t_logits = teacher(x)
        loss = kd_loss_fn(s_logits, t_logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
        preds += s_logits.argmax(dim=1).cpu().tolist()
        trues += y.cpu().tolist()
    loss = running_loss / len(loader.dataset)
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average='macro')
    return loss, acc, f1

In [9]:

# --------------------------
# HBFP-style bookkeeping: record per-filter L1 norms during training
# --------------------------
def get_conv_filters(module):
    # returns list of (module, name) for Conv2d modules
    convs = []
    for name, m in module.named_modules():
        if isinstance(m, nn.Conv2d):
            convs.append((m, name))
    return convs

def record_filter_l1_history(model, conv_hist: Dict[str, List[np.ndarray]]):
    # conv_hist: mapping layer_name -> list of per-epoch L1 arrays (num_filters,)
    for module, name in get_conv_filters(model):
        w = module.weight.detach().cpu().numpy()  # shape (out_ch, in_ch, k,k)
        l1_per_filter = np.sum(np.abs(w), axis=(1,2,3))
        if name not in conv_hist:
            conv_hist[name] = []
        conv_hist[name].append(l1_per_filter.copy())

def compute_pairwise_summed_diff(conv_hist: Dict[str, List[np.ndarray]]):
    # For each conv layer, compute nC2 pairwise sum over epochs of absolute differences
    # Return per-layer list of tuples (i,j,sumdiff)
    selection = {}
    for name, history in conv_hist.items():
        # history is list of arrays (num_filters,)
        arr = np.stack(history, axis=0)  # (epochs, num_filters)
        epochs = arr.shape[0]
        num_filters = arr.shape[1]
        pair_list = []
        # compute D_{i,j} = sum_t |l1_i(t) - l1_j(t)|
        # O(n^2 * epochs) but conv filters often manageable
        for i in range(num_filters):
            for j in range(i+1, num_filters):
                diff = np.sum(np.abs(arr[:, i] - arr[:, j]))
                pair_list.append((i, j, diff))
        selection[name] = pair_list
    return selection


In [10]:
# --------------------------
# Pruning helpers: pick M% pairs with smallest D and prune the weaker filter (smaller final-l1)
# --------------------------
def hbfp_select_pairs_and_prune(model, conv_hist, prune_fraction_per_layer=0.2, device='cpu', regularizer_lambda=1.0, optimize_epochs=3, lr=1e-4, dummy_loader=None):
    """
    conv_hist: recorded history (layer_name -> list of per-epoch L1 arrays)
    prune_fraction_per_layer: fraction of filters to remove (percentage of filters)
    Approach:
      - For each conv layer, compute pairwise D values over training history
      - Select top M% pairs with smallest D (most similar)
      - For each selected pair, determine the 'weaker' filter (smaller current L1)
      - Optionally optimize the model with a regularizer that reduces |l1_i - l1_j| for selected pairs
      - Apply structured channel pruning (zero-out entire out-channel) for the weaker filters
    """
    # compute pairwise sums
    selection = compute_pairwise_summed_diff(conv_hist)
    # Build set of channels to prune
    to_prune = {}  # layer_name -> list of output channel indices
    # choose pairs to match fraction
    for module, name in get_conv_filters(model):
        if name not in selection:
            continue
        pairs = selection[name]  # list (i,j,diff)
        if not pairs:
            continue
        pairs_sorted = sorted(pairs, key=lambda x: x[2])  # ascending diff (most similar first)
        num_filters = module.weight.shape[0]
        # number of filters to prune
        prune_k = int(np.round(prune_fraction_per_layer * num_filters))
        # each pair yields one prune, so number of pairs to select = prune_k
        selected_pairs = pairs_sorted[:max(0, prune_k)]
        # decide for each pair which filter to prune by looking at the last epoch l1
        last_epoch_arr = np.array(conv_hist[name][-1])
        pr_indices = []
        for (i,j,d) in selected_pairs:
            # prune the filter with smaller last_l1
            if last_epoch_arr[i] < last_epoch_arr[j]:
                pr = i
            else:
                pr = j
            pr_indices.append(pr)
        to_prune[name] = sorted(list(set(pr_indices)))
    # Optimization regularizer: reduce |l1_i - l1_j| for selected pairs (approximate)
    # We'll implement a small training loop over dummy_loader using a custom regularizer added to CE loss
    if regularizer_lambda > 0 and dummy_loader is not None:
        # find selected pairs per layer again but keep both indices
        selected_pairs_map = {}
        for module, name in get_conv_filters(model):
            if name not in selection:
                continue
            pairs_sorted = sorted(selection[name], key=lambda x: x[2])
            num_filters = module.weight.shape[0]
            prune_k = int(np.round(prune_fraction_per_layer * num_filters))
            sel_pairs = [(i,j) for (i,j,_) in pairs_sorted[:max(0, prune_k)]]
            if sel_pairs:
                selected_pairs_map[name] = sel_pairs
        if selected_pairs_map:
            # small optimization: run a few epochs with CE + reg to increase similarity
            opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
            ce = nn.CrossEntropyLoss()
            model.train()
            for e in range(optimize_epochs):
                for xb, yb in dummy_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad()
                    logits = model(xb)
                    loss = ce(logits, yb)
                    # build regularizer term
                    reg = 0.0
                    for module, name in get_conv_filters(model):
                        if name not in selected_pairs_map:
                            continue
                        w = module.weight  # [out_ch, in_ch, k,k]
                        # compute l1 per filter
                        l1 = w.abs().view(w.size(0), -1).sum(dim=1)
                        for (i,j) in selected_pairs_map[name]:
                            reg = reg + torch.abs(l1[i] - l1[j])
                    loss = loss + regularizer_lambda * reg
                    loss.backward()
                    opt.step()
    # Apply structured pruning (zero-out the selected output channels)
    # We'll use torch.nn.utils.prune.ln_structured to prune channels by L1 (but we specify indices manually)
    pruned_count = 0
    for module, name in get_conv_filters(model):
        if name not in to_prune:
            continue
        idxs = to_prune[name]
        if len(idxs) == 0:
            continue
        # prune each filter index by zeroing out the corresponding output channel weights and bias
        # torch prune doesn't accept explicit indices in ln_structured, so we do manual masking:
        with torch.no_grad():
            w = module.weight.data  # shape (out_ch, in_ch, k,k)
            mask = torch.ones_like(w)
            for idx in idxs:
                mask[idx].zero_()
            module.weight.data.mul_(mask)
            if module.bias is not None:
                bmask = torch.ones_like(module.bias.data)
                for idx in idxs:
                    bmask[idx] = 0.0
                module.bias.data.mul_(bmask)
        pruned_count += len(idxs)
    return pruned_count, to_prune